# YOLO11s Ball Detection Training with ClearML on AWS SageMaker

This notebook trains a YOLO11s object detection model for ball detection using a YOLO-format dataset stored in ClearML. It follows the structure of the provided `train_ball.py` script but converts it into a cell-by-cell AWS SageMaker Jupyter workflow.

The notebook does **not** hardcode ClearML secret keys. Paste them securely when prompted.

## Install required packages

Run this once at the start of the SageMaker session.

In [1]:
!pip install -U ultralytics clearml opencv-python matplotlib pandas pyyaml

## Set ClearML credentials securely

Paste your ClearML keys into the prompts. The values will not be printed in the notebook output.

In [2]:
import shutil
import pathlib

clearml_cache_dirs = [
    pathlib.Path.home() / ".clearml" / "cache",
    pathlib.Path.home() / ".clearml" / "vcs-cache",
    pathlib.Path.home() / ".clearml" / "pip-download-cache",
]

for cache_dir in clearml_cache_dirs:
    if cache_dir.exists():
        print("Removing:", cache_dir)
        shutil.rmtree(cache_dir)
    else:
        print("Not found:", cache_dir)

print("ClearML cache cleanup complete.")

Removing: /home/sagemaker-user/.clearml/cache
Not found: /home/sagemaker-user/.clearml/vcs-cache
Not found: /home/sagemaker-user/.clearml/pip-download-cache
ClearML cache cleanup complete.


In [4]:
!df -h

Filesystem      Size  Used Avail Use% Mounted on
overlay          37G  717M   37G   2% /
tmpfs            64M     0   64M   0% /dev
shm             1.8G   40K  1.8G   1% /dev/shm
/dev/nvme0n1p1  180G  144G   37G  80% /usr/bin/nvidia-smi
/dev/nvme2n1     50G   14G   37G  27% /home/sagemaker-user
/dev/nvme1n1    117G  863M  116G   1% /mnt/sagemaker-nvme
127.0.0.1:/     8.0E  145G  8.0E   1% /mnt/custom-file-systems/efs/fs-02d21030926db1df2_fsap-0aa957d6f6a68347e
tmpfs           7.8G   12K  7.8G   1% /proc/driver/nvidia
tmpfs           3.1G  9.1M  3.1G   1% /run/nvidia-persistenced/socket
tmpfs           7.8G     0  7.8G   0% /proc/acpi
tmpfs           7.8G     0  7.8G   0% /sys/firmware


In [5]:
import os
from getpass import getpass

os.environ["CLEARML_WEB_HOST"] = "https://app.clear.ml/"
os.environ["CLEARML_API_HOST"] = "https://api.clear.ml"
os.environ["CLEARML_FILES_HOST"] = "https://files.clear.ml"

os.environ["CLEARML_API_ACCESS_KEY"] = getpass("RPLZH4X3O0CQTDJIXADMPV9GDSAGQT")
os.environ["CLEARML_API_SECRET_KEY"] = getpass("CqNOhwssMDxNXkCqRaWAA6BertVIA-8-4CAffYQ62xFg79N5XWyfkeyx4MvucJfm8Rc")

print("ClearML environment variables are set.")

RPLZH4X3O0CQTDJIXADMPV9GDSAGQT ········
CqNOhwssMDxNXkCqRaWAA6BertVIA-8-4CAffYQ62xFg79N5XWyfkeyx4MvucJfm8Rc ········


ClearML environment variables are set.


## Imports

In [6]:
import pathlib
import shutil
import time
import json
import random
from datetime import datetime

import torch
import pandas as pd
import matplotlib.pyplot as plt
import yaml

from clearml import Task, OutputModel, Dataset
from ultralytics import YOLO

## Configuration

This notebook uses YOLO11s pretrained weights instead of the original `yolo26n.yaml`. For small-ball detection, keep `imgsz` reasonably high. Start with `640`; increase to `960` if the ball is missed too often and the GPU has enough memory.

In [8]:
SEED = 42

PROJECT_NAME = "PitchSense_v2"
TASK_NAME = "yolo11s_ball_detection"

CLEARML_DATASET_ID = "300ca95787984633bc04cf2155a4c968"

MODEL_NAME = "yolo11s.pt"

EPOCHS = 200
IMGSZ = 640
BATCH = 16
PATIENCE = 25

DEVICE = 0 if torch.cuda.is_available() else "cpu"

OUTPUT_ROOT = pathlib.Path.home() / "ball_detection_outputs" / "yolo11s_ball_detection"
SAVE_DIR = OUTPUT_ROOT / "saved_models"

RUN_NAME = "yolo11s_ball_detection"

CLEAN_OUTPUT_DIR = True

random.seed(SEED)
torch.manual_seed(SEED)

print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
print("Training device:", DEVICE)
print("Output root:", OUTPUT_ROOT)

CUDA available: True
CUDA device count: 1
Training device: 0
Output root: /home/sagemaker-user/ball_detection_outputs/yolo11s_ball_detection


## Timer helper

This prints progress timestamps so you can see where the notebook is spending time.

In [9]:
class StepTimer:
    def __init__(self):
        self.start_time = time.time()

    def log(self, message):
        elapsed = time.time() - self.start_time
        minutes = int(elapsed // 60)
        seconds = int(elapsed % 60)
        print(f"[{minutes:02d}m {seconds:02d}s] {message}")


timer = StepTimer()

## Dataset loading helper

This pulls the pinned ClearML dataset and expects a `dataset.yaml` file inside the local dataset copy.

In [10]:
def pull_yolo_dataset_from_clearml(clearml_dataset_id):
    timer.log("Pulling YOLO dataset from ClearML.")

    clearml_dataset = Dataset.get(dataset_id=clearml_dataset_id)
    local_dataset_path = pathlib.Path(clearml_dataset.get_local_copy())
    yaml_path = local_dataset_path / "dataset.yaml"

    if not yaml_path.exists():
        possible_yaml_files = list(local_dataset_path.rglob("*.yaml")) + list(local_dataset_path.rglob("*.yml"))
        print("Available YAML files:")
        for path in possible_yaml_files:
            print(path)

        raise FileNotFoundError(
            f"dataset.yaml was not found at {yaml_path}. "
            "If your dataset YAML has a different name, update yaml_path manually."
        )

    print("ClearML dataset pulled successfully.")
    print("ClearML dataset ID:", clearml_dataset.id)
    print("Local dataset path:", local_dataset_path)
    print("YOLO dataset YAML:", yaml_path)

    return clearml_dataset, yaml_path

## Output directory helper

In [12]:
def prepare_output_directory():
    if CLEAN_OUTPUT_DIR and OUTPUT_ROOT.exists():
        shutil.rmtree(OUTPUT_ROOT)

    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    print("Output directory ready:", OUTPUT_ROOT)

## Model saving helper

In [13]:
def copy_best_weights(save_dir, run_name, model_name_for_file):
    best_weights = save_dir / run_name / "weights" / "best.pt"
    final_model_path = save_dir / f"{model_name_for_file}_best.pt"

    if best_weights.exists():
        shutil.copy2(best_weights, final_model_path)
        print("Best model copied to:", final_model_path)
        return final_model_path

    print("Warning: best.pt was not found after training.")
    return None

## Results helpers

In [14]:
def read_training_results(run_dir):
    results_csv = run_dir / "results.csv"

    if not results_csv.exists():
        print("results.csv not found:", results_csv)
        return None

    df = pd.read_csv(results_csv)
    df.columns = [col.strip() for col in df.columns]
    display(df.tail())

    return df


def plot_training_curves(results_df):
    if results_df is None:
        print("No results dataframe available.")
        return

    metric_candidates = [
        "metrics/mAP50(B)",
        "metrics/mAP50-95(B)",
        "metrics/precision(B)",
        "metrics/recall(B)",
        "train/box_loss",
        "train/cls_loss",
        "val/box_loss",
        "val/cls_loss",
    ]

    available_metrics = [metric for metric in metric_candidates if metric in results_df.columns]

    for metric in available_metrics:
        plt.figure(figsize=(8, 5))
        plt.plot(results_df["epoch"], results_df[metric])
        plt.xlabel("Epoch")
        plt.ylabel(metric)
        plt.title(metric)
        plt.grid(True)
        plt.show()

## Start ClearML task

In [15]:
timer.log("Starting ClearML task.")

task = Task.init(
    project_name=PROJECT_NAME,
    task_name=TASK_NAME,
    task_type=Task.TaskTypes.training,
)

config = {
    "model_name": MODEL_NAME,
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "patience": PATIENCE,
    "seed": SEED,
    "device": str(DEVICE),
    "output_root": str(OUTPUT_ROOT),
    "save_dir": str(SAVE_DIR),
    "clearml_dataset_id": CLEARML_DATASET_ID,
    "run_name": RUN_NAME,
}

task.connect(config)

print("ClearML task created.")
print("Task ID:", task.id)

 28s] Starting ClearML task.
ClearML Task: created new task id=f6733e355bf2487197fa15ad316fdf17
2026-05-24 18:44:33,187 - clearml.Task - INFO - Storing jupyter notebook directly as code


E0000 00:00:1779648276.380046    2519 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779648276.385779    2519 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


ClearML results page: https://app.clear.ml/projects/68bbb295f3954257afaa4dfdb9b2811a/experiments/f6733e355bf2487197fa15ad316fdf17/output/log
ClearML task created.
Task ID: f6733e355bf2487197fa15ad316fdf17
ClearML Monitor: Could not detect iteration reporting, falling back to iterations as seconds-from-start


## Load YOLO dataset from ClearML

In [16]:
clearml_dataset, yaml_path = pull_yolo_dataset_from_clearml(CLEARML_DATASET_ID)

task.upload_artifact("dataset_yaml_path", artifact_object=str(yaml_path))
task.upload_artifact("clearml_dataset_id", artifact_object=clearml_dataset.id)

prepare_output_directory()

 13s] Pulling YOLO dataset from ClearML.
2026-05-24 18:45:09,951 - clearml - INFO - Dataset.get() did not specify alias. Dataset information will not be automatically logged in ClearML Server.
2026-05-24 18:45:16,671 - clearml.storage - INFO - Downloading: 0.00MB from https://files.clear.ml/PitchSense_v2/.datasets/Soccernet_ball_subset/Soccernet_ball_subset.300ca95787984633bc04cf2155a4c968/artifacts/state/state.json


                                             0% | 0.00/0.0 MB [00:00<?, ?MB/s]: 

2026-05-24 18:45:16,675 - clearml.storage - INFO - Downloading: 8.08MB / 0.00MB @ 3.53MBs from https://files.clear.ml/PitchSense_v2/.datasets/Soccernet_ball_subset/Soccernet_ball_subset.300ca95787984633bc04cf2155a4c968/artifacts/state/state.json
2026-05-24 18:45:16,677 - clearml.storage - INFO - Downloading: 8.11MB / 0.00MB @ 35.66MBs from https://files.clear.ml/PitchSense_v2/.datasets/Soccernet_ball_subset/Soccernet_ball_subset.300ca95787984633bc04cf2155a4c968/artifacts/state/state.json
2026-05-24 18:45:16,679 - clearml.storage - INFO - Downloading: 8.14MB / 0.00MB @ 25.62MBs from https://files.clear.ml/PitchSense_v2/.datasets/Soccernet_ball_subset/Soccernet_ball_subset.300ca95787984633bc04cf2155a4c968/artifacts/state/state.json
2026-05-24 18:45:16,681 - clearml.storage - INFO - Downloading: 8.17MB / 0.00MB @ 33.04MBs from https://files.clear.ml/PitchSense_v2/.datasets/Soccernet_ball_subset/Soccernet_ball_subset.300ca95787984633bc04cf2155a4c968/artifacts/state/state.json
2026-05-24 18

                                             0% | 0.00/0.0 MB [00:00<?, ?MB/s]: 

2026-05-24 18:45:17,421 - clearml.storage - INFO - Downloaded 0.00 MB successfully from https://files.clear.ml/PitchSense_v2/.datasets/Soccernet_ball_subset/Soccernet_ball_subset.300ca95787984633bc04cf2155a4c968/artifacts/state/state.json , saved to /home/sagemaker-user/.clearml/cache/storage_manager/datasets/b37d2a9b44abf083b0ca0e4fa7aacda0.state.json


...

2026-05-24 19:07:07,230 - clearml.storage - INFO - Downloaded 430.10 MB successfully from https://files.clear.ml/PitchSense_v2/.datasets/Soccernet_ball_subset/Soccernet_ball_subset.300ca95787984633bc04cf2155a4c968/artifacts/data_034/dataset.300ca95787984633bc04cf2155a4c968.9oq6ujfd.zip , saved to /home/sagemaker-user/.clearml/cache/storage_manager/datasets/664dc8e4779db855cb588b4719fc9f07.dataset.300ca95787984633bc04cf2155a4c968.9oq6ujfd.zip


2026-05-24 19:07:10,243 - clearml.storage - INFO - Downloading: 430.31MB from https://files.clear.ml/PitchSense_v2/.datasets/Soccernet_ball_subset/Soccernet_ball_subset.300ca95787984633bc04cf2155a4c968/artifacts/data_035/dataset.300ca95787984633bc04cf2155a4c968.snr1yp4k.zip


████████████████████████████ 100% | 430.31/430.31 MB [00:30<-00:00, 13.90MB/s]: 

2026-05-24 19:07:41,194 - clearml.storage - INFO - Downloaded 430.31 MB successfully from https://files.clear.ml/PitchSense_v2/.datasets/Soccernet_ball_subset/Soccernet_ball_subset.300ca95787984633bc04cf2155a4c968/artifacts/data_035/dataset.300ca95787984633bc04cf2155a4c968.snr1yp4k.zip , saved to /home/sagemaker-user/.clearml/cache/storage_manager/datasets/02c53bf65060bb8c02a8a3e976393cf4.dataset.300ca95787984633bc04cf2155a4c968.snr1yp4k.zip


ClearML dataset pulled successfully.
ClearML dataset ID: 300ca95787984633bc04cf2155a4c968
Local dataset path: /home/sagemaker-user/.clearml/cache/storage_manager/datasets/ds_300ca95787984633bc04cf2155a4c968
YOLO dataset YAML: /home/sagemaker-user/.clearml/cache/storage_manager/datasets/ds_300ca95787984633bc04cf2155a4c968/dataset.yaml
Output directory ready: /home/sagemaker-user/ball_detection_outputs/yolo11s_ball_detection


In [19]:
ls

__pycache__/   train_players.py                                yolo26n.pt
constants.py   train_yolo26_pose_scratch.py
train_ball.py  yolo11s_ball_detection_clearml_sagemaker.ipynb


## Inspect dataset YAML

In [17]:
with open(yaml_path, "r") as file:
    dataset_config = yaml.safe_load(file)

print(json.dumps(dataset_config, indent=2))

{
  "path": "/home/aanil/Data/aanil/side/yolo/outputs/yolo_26n_ball_200epochs",
  "train": "images/train",
  "val": "images/val",
  "test": "images/test",
  "names": {
    "0": "ball"
  }
}


## Train YOLO11s for ball detection

In [19]:
import pathlib
import yaml
import os

print("yaml_path:", yaml_path)
print("dataset root:", yaml_path.parent)

print("\nTop-level files/folders:")
for p in yaml_path.parent.iterdir():
    print(p)

yaml_path: /home/sagemaker-user/.clearml/cache/storage_manager/datasets/ds_300ca95787984633bc04cf2155a4c968/dataset.yaml
dataset root: /home/sagemaker-user/.clearml/cache/storage_manager/datasets/ds_300ca95787984633bc04cf2155a4c968

Top-level files/folders:
/home/sagemaker-user/.clearml/cache/storage_manager/datasets/ds_300ca95787984633bc04cf2155a4c968/images
/home/sagemaker-user/.clearml/cache/storage_manager/datasets/ds_300ca95787984633bc04cf2155a4c968/labels
/home/sagemaker-user/.clearml/cache/storage_manager/datasets/ds_300ca95787984633bc04cf2155a4c968/dataset.yaml


In [20]:
with open(yaml_path, "r") as f:
    data_yaml = yaml.safe_load(f)

data_yaml

{'path': '/home/aanil/Data/aanil/side/yolo/outputs/yolo_26n_ball_200epochs',
 'train': 'images/train',
 'val': 'images/val',
 'test': 'images/test',
 'names': {0: 'ball'}}

In [21]:
import pathlib
import yaml

dataset_root = yaml_path.parent

with open(yaml_path, "r") as f:
    data_yaml = yaml.safe_load(f)

print("Original YAML:")
print(data_yaml)

fixed_yaml = data_yaml.copy()

fixed_yaml["path"] = str(dataset_root)

# Force YOLO paths to be relative to the current ClearML dataset folder
fixed_yaml["train"] = "images/train"
fixed_yaml["val"] = "images/val"

# Only keep test if it actually exists
if (dataset_root / "images" / "test").exists():
    fixed_yaml["test"] = "images/test"
else:
    fixed_yaml.pop("test", None)

fixed_yaml_path = dataset_root / "dataset_sagemaker_fixed.yaml"

with open(fixed_yaml_path, "w") as f:
    yaml.safe_dump(fixed_yaml, f, sort_keys=False)

print("\nFixed YAML saved to:")
print(fixed_yaml_path)

print("\nFixed YAML:")
print(fixed_yaml)

Original YAML:
{'path': '/home/aanil/Data/aanil/side/yolo/outputs/yolo_26n_ball_200epochs', 'train': 'images/train', 'val': 'images/val', 'test': 'images/test', 'names': {0: 'ball'}}

Fixed YAML saved to:
/home/sagemaker-user/.clearml/cache/storage_manager/datasets/ds_300ca95787984633bc04cf2155a4c968/dataset_sagemaker_fixed.yaml

Fixed YAML:
{'path': '/home/sagemaker-user/.clearml/cache/storage_manager/datasets/ds_300ca95787984633bc04cf2155a4c968', 'train': 'images/train', 'val': 'images/val', 'test': 'images/test', 'names': {0: 'ball'}}


In [22]:
print("Train images exists:", (dataset_root / "images" / "train").exists())
print("Val images exists:", (dataset_root / "images" / "val").exists())
print("Test images exists:", (dataset_root / "images" / "test").exists())

print("Train label folder exists:", (dataset_root / "labels" / "train").exists())
print("Val label folder exists:", (dataset_root / "labels" / "val").exists())
print("Test label folder exists:", (dataset_root / "labels" / "test").exists())

print("Number of train images:", len(list((dataset_root / "images" / "train").glob("*"))))
print("Number of val images:", len(list((dataset_root / "images" / "val").glob("*"))))

Train images exists: True
Val images exists: True
Test images exists: True
Train label folder exists: True
Val label folder exists: True
Test label folder exists: True
Number of train images: 36338
Number of val images: 6412


In [ ]:
model = YOLO(MODEL_NAME)

timer.log("Starting YOLO11s training with fixed SageMaker dataset YAML.")

train_results = model.train(
    data=str(fixed_yaml_path),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project=str(SAVE_DIR),
    name=RUN_NAME,
    pretrained=True,
    patience=PATIENCE,
    seed=SEED,
    exist_ok=True,
    verbose=True,
)

timer.log("Training finished.")

 12s] Starting YOLO11s training with fixed SageMaker dataset YAML.
Ultralytics 8.4.53 🚀 Python-3.12.9 torch-2.6.0 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/sagemaker-user/.clearml/cache/storage_manager/datasets/ds_300ca95787984633bc04cf2155a4c968/dataset_sagemaker_fixed.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, 

2026/05/24 19:13:51 INFO mlflow.tracking.fluent: Experiment with name '/home/sagemaker-user/ball_detection_outputs/yolo11s_ball_detection/saved_models' does not exist. Creating a new experiment.
2026/05/24 19:13:51 INFO mlflow.bedrock: Enabled auto-tracing for Bedrock. Note that MLflow can only trace boto3 service clients that are created after this call. If you have already created one, please recreate the client by calling `boto3.client`.
2026/05/24 19:13:51 INFO mlflow.tracking.fluent: Autologging successfully enabled for boto3.
/opt/conda/lib/python3.12/site-packages/mlflow/fastai/__init__.py:68: FutureWarning:

fastai flavor is deprecated and will be removed in MLflow 3.0.

2026/05/24 19:13:51 WARNING mlflow.utils.autologging_utils: MLflow fastai autologging is known to be compatible with 2.4.1 <= fastai <= 2.7.19, but the installed version is 2.8.7. If you encounter errors during autologging, try upgrading / downgrading fastai to a compatible version, or try upgrading MLflow.
202

MLflow: logging run_id(39d2d4638a6546a6acd8b2dcc16e6e36) to runs/mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs/mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 4 dataloader workers
Logging results to /home/sagemaker-user/ball_detection_outputs/yolo11s_ball_detection/saved_models/yolo11s_ball_detection
Starting training for 200 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/200      4.05G      3.582      25.13     0.8046         16        640: 6% ╸─────────── 144/2272 2.4it/s 1:11<14:521

## Save and upload best model to ClearML

In [24]:
final_model_path = copy_best_weights(
    save_dir=SAVE_DIR,
    run_name=RUN_NAME,
    model_name_for_file="yolo11s_ball_detection",
)

if final_model_path is not None and final_model_path.exists():
    output_model = OutputModel(task=task, name="yolo11s_ball_detection_best")
    output_model.update_weights(weights_filename=str(final_model_path))

    task.upload_artifact("best_model_path", artifact_object=str(final_model_path))
    task.upload_artifact("best_model_file", artifact_object=str(final_model_path))

    print("Best model uploaded to ClearML.")
else:
    print("Best model was not uploaded because best.pt was not found.")

Best model copied to: /home/sagemaker-user/ball_detection_outputs/yolo11s_ball_detection/saved_models/yolo11s_ball_detection_best.pt
2026-05-26 11:54:46,256 - clearml.model - INFO - No output storage destination defined, registering local model /home/sagemaker-user/ball_detection_outputs/yolo11s_ball_detection/saved_models/yolo11s_ball_detection_best.pt
Best model uploaded to ClearML.


## Read and plot training results

In [25]:
run_dir = SAVE_DIR / RUN_NAME

results_df = read_training_results(run_dir)
plot_training_curves(results_df)

if results_df is not None:
    task.upload_artifact("training_results_csv", artifact_object=str(run_dir / "results.csv"))

,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2,lr/pg3,lr/pg4,lr/pg5,lr/pg6,lr/pg7
195,196,142567.0,2.11849,0.66674,0.66328,0.80791,0.64636,0.69170,0.31932,2.47902,0.83710,0.65220,0.001043,0.000347,0.001043,0.000347,0.001043,0.000347,0.001043,0.000347
196,197,143280.0,2.11434,0.66385,0.66167,0.80557,0.64816,0.69118,0.31964,2.47727,0.83622,0.65224,0.000894,0.000298,0.000894,0.000298,0.000894,0.000298,0.000894,0.000298
197,198,143992.0,2.10852,0.66119,0.66167,0.80485,0.65060,0.69158,0.31963,2.47787,0.83630,0.65225,0.000745,0.000249,0.000745,0.000249,0.000745,0.000249,0.000745,0.000249
198,199,144698.0,2.10781,0.66119,0.66138,0.80519,0.65060,0.69135,0.31958,2.47758,0.83687,0.65229,0.000597,0.000199,0.000597,0.000199,0.000597,0.000199,0.000597,0.000199
199,200,145405.0,2.10206,0.65933,0.66366,0.80492,0.65190,0.69159,0.31933,2.47727,0.83708,0.65228,0.000449,0.000150,0.000449,0.000150,0.000449,0.000150,0.000449,0.000150


<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

...

<Figure size 800x500 with 1 Axes>

<Figure size 800x500 with 1 Axes>

<Figure size 800x500 with 1 Axes>

<Figure size 800x500 with 1 Axes>

<Figure size 800x500 with 1 Axes>

## Validate best model on validation split

In [26]:
if final_model_path is not None and final_model_path.exists():
    timer.log("Running validation evaluation using best model.")

    best_model = YOLO(str(final_model_path))

    val_results = best_model.val(
        data=str(fixed_yaml_path),
        split="val",
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        project=str(SAVE_DIR),
        name="val_eval",
        exist_ok=True,
        verbose=True,
    )

    task.upload_artifact("validation_results", artifact_object=str(val_results))
else:
    print("Skipping validation because best model is unavailable.")

[2470m 58s] Running validation evaluation using best model.
Ultralytics 8.4.53 🚀 Python-3.12.9 torch-2.6.0 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 270.7±121.9 MB/s, size: 246.4 KB)
val: Scanning /home/sagemaker-user/.clearml/cache/storage_manager/datasets/ds_300ca95787984633bc04cf2155a4c968/labels/val.cache... 6412 images, 363 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 6412/6412 2.4Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 401/401 7.4it/s 54.2s<0.1s
                   all       6412       6162       0.81      0.655      0.698      0.329
Speed: 0.7ms preprocess, 4.7ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to /home/sagemaker-user/ball_detection_outputs/yolo11s_ball_detection/saved_models/val_eval


## Test evaluation if a test split exists

In [27]:
if final_model_path is not None and final_model_path.exists():
    timer.log("Trying test evaluation using best model.")

    best_model = YOLO(str(final_model_path))

    try:
        test_results = best_model.val(
            data=str(fixed_yaml_path),
            split="test",
            imgsz=IMGSZ,
            batch=BATCH,
            device=DEVICE,
            project=str(SAVE_DIR),
            name="test_eval",
            exist_ok=True,
            verbose=True,
        )

        task.upload_artifact("test_results", artifact_object=str(test_results))

    except Exception as error:
        print("Test evaluation failed. This usually means the dataset YAML has no test split.")
        print("Error:", error)
else:
    print("Skipping test evaluation because best model is unavailable.")

[2471m 58s] Trying test evaluation using best model.
Ultralytics 8.4.53 🚀 Python-3.12.9 torch-2.6.0 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 124.7±6.7 MB/s, size: 287.6 KB)
val: Scanning /home/sagemaker-user/.clearml/cache/storage_manager/datasets/ds_300ca95787984633bc04cf2155a4c968/labels/test... 36750 images, 2946 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 36750/36750 1.0Kit/s 36.2s<0.1ss
val: New cache created: /home/sagemaker-user/.clearml/cache/storage_manager/datasets/ds_300ca95787984633bc04cf2155a4c968/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2297/2297 8.1it/s 4:42<0.1ss
                   all      36750      34129      0.539      0.304      0.275     0.0877
Speed: 0.6ms preprocess, 4.7ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to /home/sagemaker-user/b

## Prepare sample images for inference

In [28]:
dataset_root = pathlib.Path(dataset_config.get("path", yaml_path.parent))

val_path = dataset_config.get("val")
if val_path is None:
    raise ValueError("No validation path found in dataset YAML.")

val_images_path = pathlib.Path(val_path)
if not val_images_path.is_absolute():
    val_images_path = dataset_root / val_images_path

image_files = []
for extension in ["*.jpg", "*.jpeg", "*.png"]:
    image_files.extend(list(val_images_path.rglob(extension)))

print("Validation images found:", len(image_files))

sample_images = image_files[:5]
sample_images

Validation images found: 0


[]

## Run sample inference

In [29]:
if final_model_path is not None and final_model_path.exists() and len(sample_images) > 0:
    timer.log("Running sample inference.")

    best_model = YOLO(str(final_model_path))

    inference_output_dir = SAVE_DIR / "sample_inference"
    inference_output_dir.mkdir(parents=True, exist_ok=True)

    predictions = best_model.predict(
        source=[str(path) for path in sample_images],
        imgsz=IMGSZ,
        conf=0.25,
        device=DEVICE,
        save=True,
        project=str(inference_output_dir),
        name="predictions",
        exist_ok=True,
    )

    task.upload_artifact("sample_inference_dir", artifact_object=str(inference_output_dir))

    print("Sample inference completed.")
    print("Saved to:", inference_output_dir)
else:
    print("Skipping inference because model or sample images are unavailable.")

Skipping inference because model or sample images are unavailable.


## Close ClearML task

In [30]:
task.upload_artifact("output_root", artifact_object=str(OUTPUT_ROOT))

timer.log("All training, evaluation, and artifact upload steps completed.")

task.close()
print("ClearML task closed.")

[2477m 29s] All training, evaluation, and artifact upload steps completed.
ClearML task closed.


## Resume training later if needed

Only run this section if training stopped and `last.pt` exists.

In [ ]:
last_checkpoint = SAVE_DIR / RUN_NAME / "weights" / "last.pt"

if last_checkpoint.exists():
    print("Resuming from:", last_checkpoint)

    model = YOLO(str(last_checkpoint))

    resumed_results = model.train(
        data=str(fixed_yaml_path),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        project=str(SAVE_DIR),
        name=RUN_NAME,
        resume=True,
        exist_ok=True,
        verbose=True,
    )
else:
    print("No last.pt checkpoint found at:", last_checkpoint)